# Classifier and its evidence — Plots 5, 6, 7

Decides whether a sound becomes a highlighted pin. Labels: `knock_tap`, `voice`, `impact`, `other`.

**Evaluation is leave-one-session-out:** each session (one sitting: one person, one surface, one room) is held out in turn; the model is trained on the others (plus noisy copies of *their* clips only) and tested on the held-out clips. Every number below is on sessions the model never saw. A random split by clip would put near-identical clips in train and test and inflate the score.

Real clips need no code change: drop them in `data/clips/<label>/<session>_<n>.npz` (keys `audio`, `fs_hz`, `label`, `session`, `environment`) and rerun all.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import classify as K, features as F

AUGMENT_SNRS_DB = (20, 10, 5)      # noisy copies of TRAINING clips (stay in their session's split)
clips = K.load_clips()
print(f"{len(clips)} clips | sessions: {sorted(clips.session.unique())}")
print(clips.groupby(['label', 'session']).size().unstack(fill_value=0))
print("MFCC features:", "on (librosa)" if F.librosa else "off (librosa not installed)")

In [ ]:
feat = K.featurize(clips, augment_snrs=AUGMENT_SNRS_DB)
res = K.loso_eval(feat)
results = {}
for name in ("rules", "model"):
    cm, table, resp = K.report(res.label, res[f"pred_{name}"])
    results[name] = (cm, table, resp)
    print(f"=== {name}: person missed {resp['person_missed']:.0%}, noise called a person {resp['false_alarm']:.0%} "
          f"| knock recall {resp['knock_recall']:.0%}, voice recall {resp['voice_recall']:.0%}")
    display(table)

In [ ]:
# Which one wins? The responder metric first: how often a real person is missed.
r, m = results["rules"][2], results["model"][2]
winner = "model" if (m["person_missed"], m["false_alarm"]) < (r["person_missed"], r["false_alarm"]) else "rules"
print(f"WINNER: {winner}. Person missed: rules {r['person_missed']:.0%} vs model {m['person_missed']:.0%}; "
      f"false alarms: rules {r['false_alarm']:.0%} vs model {m['false_alarm']:.0%}.")
k = res[res.label == "knock_tap"]
print("\nKnock recall per held-out session (a surface the model never heard scores low):")
print(k.assign(hit=k[f"pred_{winner}"] == "knock_tap").groupby("session").hit.agg(['sum', 'count']))

In [ ]:
cm, table, resp = results[winner]
n_k = int(cm[0].sum()); hit_k = int(cm[0, 0])
title = (f"Knocks recognized in {hit_k} of {n_k} held-out clips\nA person is missed "
         f"{resp['person_missed']:.0%} of the time, noise called a person {resp['false_alarm']:.0%} ({winner})")
fig = K.plot_confusion(cm, title)
fig.savefig(ROOT / "docs/figures/plot5_confusion.png", dpi=130); plt.show()

In [ ]:
fig = K.plot_gallery(clips)
fig.savefig(ROOT / "docs/figures/plot6_spectrograms.png", dpi=110); plt.show()

In [ ]:
raw = feat[~feat.augmented]
fig = K.plot_scatter(raw, "duration_s", "pitch_strength",
                     "Voices sit high on pitch strength; knocks and impacts are short and unpitched")
fig.savefig(ROOT / "docs/figures/plot7_feature_scatter.png", dpi=130); plt.show()

### Train the final model on all sessions
`pipeline.py` uses `models/classifier.joblib` automatically once it exists (else its duration rules).

In [ ]:
K.train_final(feat)
print("saved", K.MODEL_PATH.relative_to(ROOT))